<a href="https://colab.research.google.com/github/vinhdo19111999-hash/Start/blob/main/law.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install python-docx pandas scikit-learn openpyxl
!pip install docx2txt

In [ ]:
import re
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

import docx2txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_path_word = '/content/drive/MyDrive/Colab Notebooks/NLP/law_retrieval/data/84_2015_QH13_281961.docx'

In [ ]:
from docx import Document

def extract_all_text_from_docx(file_path):
    try:
        # docx2txt tự động đọc toàn bộ paragraph và table theo đúng thứ tự hiển thị
        full_text = docx2txt.process(file_path)
        return full_text
    except Exception as e:
        print(f"Lỗi khi đọc file: {e}")
        return None

# Đọc lại file của bạn
file_path_word = '/content/drive/MyDrive/Colab Notebooks/NLP/law_retrieval/data/84_2015_QH13_281961.docx'
full_text = extract_all_text_from_docx(file_path_word)

if full_text:
    print(f"Đã đọc file thành công, độ dài văn bản: {len(full_text)} ký tự")
    print("--- Văn bản mẫu ---")
    print(full_text[:500])  # Sẽ in ra đầy đủ phần Quốc hội, Cộng hòa...

Đã đọc file thành công, độ dài văn bản: 134340 ký tự
--- Văn bản mẫu ---
QUỐC HỘI
-------

CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
Độc lập - Tự do - Hạnh phúc 
---------------

Luật số: 84/2015/QH13

Hà Nội, ngày 25 tháng 06 năm 2015

 

LUẬT

AN TOÀN, VỆ SINH LAO ĐỘNG

Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam;

Quốc hội ban hành Luật an toàn, vệ sinh lao động.

Chương I

QUY ĐỊNH CHUNG

Điều 1. Phạm vi điều chỉnh

Luật này quy định việc bảo đảm an toàn, vệ sinh lao động; chính sách, chế độ đối với người bị tai nạn lao động, bệnh nghề nghiệp; trách nhiệm v


In [ ]:
#Sử dụng regex để chuẩn hóa các ký tự, xử lý các lỗi font chữ thường gặp...

def standardize_text(text): #Chuẩn hóa văn bản, xử lý các lỗi font và ký tự đặc biệt
    if not text: #Kiểm tra nếu đầu vào là chuỗi rỗng, None hoặc False
        return "" #Trả về chuỗi rỗng ngay lập tức để tránh lỗi khi xử lý

    text = re.sub(r'[\x82\x84\x85\x91\x92\x93\x94\x96\x97]', ' ', text) #Thay thế các ký tự bị lỗi font thường gặp thành dấu cách

    text = re.sub(r'[ \t]+', ' ', text).strip()#Thay thế nhiều khoảng trắng/tab liền nhau thành 1 khoảng trắng, và xóa khoảng trắng ở 2 đầu chuỗi

    text = re.sub(r'[^\w\s\.\,\;\n]', ' ', text)#(Tùy chọn) Bỏ comment dòng này nếu muốn loại bỏ sạch các ký tự đặc biệt, chỉ giữ lại chữ, số, khoảng trắng và các dấu . , ;

    return text #Trả về kết quả chuỗi văn bản đã được chuẩn hóa hoàn tất

# Áp dụng chuẩn hóa cho văn bản đã đọc
if full_text:#Kiểm tra xem biến full_text (văn bản đã đọc) có chứa dữ liệu hay không
    full_text_standardized = standardize_text(full_text) #Gọi hàm chuẩn hóa và lưu kết quả vào biến full_text_standardized
    print(f"Văn bản đã được chuẩn hóa, độ dài: {len(full_text_standardized)} ký tự")#In ra màn hình xác nhận kèm tổng số lượng ký tự của văn bản mới
    print(full_text_standardized[:500])

Văn bản đã được chuẩn hóa, độ dài: 134340 ký tự
QUỐC HỘI
       

CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
Độc lập   Tự do   Hạnh phúc 
               

Luật số  84 2015 QH13

Hà Nội, ngày 25 tháng 06 năm 2015

 

LUẬT

AN TOÀN, VỆ SINH LAO ĐỘNG

Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam;

Quốc hội ban hành Luật an toàn, vệ sinh lao động.

Chương I

QUY ĐỊNH CHUNG

Điều 1. Phạm vi điều chỉnh

Luật này quy định việc bảo đảm an toàn, vệ sinh lao động; chính sách, chế độ đối với người bị tai nạn lao động, bệnh nghề nghiệp; trách nhiệm v


In [ ]:
#Phân tích cấu trúc của một văn bản luật thành các chunks (Điều, Khoản, Điểm)
def parse_law_text_to_chunks(text):
    rows = []
    current_chapter_id = None
    current_chapter_name = None
    current_article_id = None
    current_article_title = None
    current_clause_id = None
    current_item_id = None
    current_level = None
    current_text = []

    patterns = {
        'chapter': re.compile(r'^Chương\s+([IVXLCDM]+)\.\s*(.+)$', re.IGNORECASE),
        'article': re.compile(r'^Điều\s+(\d+)\.\s*(.+)$', re.IGNORECASE),
        'clause': re.compile(r'^(\d+)\.\s*(.+)$', re.IGNORECASE),
        'item': re.compile(r'^([a-zđ])\)\s*(.+)$', re.IGNORECASE),
        'clause_intro': re.compile(r'^(\d+)\.\s*$', re.IGNORECASE), # Trường hợp Khoản chỉ có số, không có nội dung trên cùng dòng
        'item_intro': re.compile(r'^([a-zđ])\)\s*$', re.IGNORECASE), # Trường hợp Điểm chỉ có ký tự, không có nội dung trên cùng dòng
        'khoan_bo_sung': re.compile(r'^(Trường hợp|Căn cứ|Chính phủ|Nhà nước|Đối với|Sau khi|Việc).*$') # Nhận diện câu bổ sung sau khoản
    }

    lines = text.split('\n')
    current_major_title = "" # Dùng để lưu tên chính của điều/khoản nếu bị tách xuống dòng

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Phát hiện Chương
        chapter_match = patterns['chapter'].match(line)
        if chapter_match:
            # Lưu chunk trước đó nếu có
            if current_text:
                rows.append({
                    'chuong_id': current_chapter_id,
                    'chuong_ten': current_chapter_name,
                    'dieu_id': current_article_id,
                    'dieu_ten': current_article_title,
                    'khoan_id': current_clause_id,
                    'diem_id': current_item_id,
                    'cap_do': current_level,
                    'noi_dung': ' '.join(current_text).strip(),
                    'full_citation': f"Điều {current_article_id} - Luật An toàn, vệ sinh lao động số 84/2015/QH13" if current_article_id else ""
                })
                current_text = []

            current_chapter_id = chapter_match.group(1)
            current_chapter_name = chapter_match.group(2).strip()
            current_article_id = None
            current_article_title = None
            current_clause_id = None
            current_item_id = None
            current_level = None
            current_major_title = ""
            continue

        # Phát hiện Điều
        article_match = patterns['article'].match(line)
        if article_match:
            # Lưu chunk trước đó nếu có
            if current_text:
                rows.append({
                    'chuong_id': current_chapter_id,
                    'chuong_ten': current_chapter_name,
                    'dieu_id': current_article_id,
                    'dieu_ten': current_article_title,
                    'khoan_id': current_clause_id,
                    'diem_id': current_item_id,
                    'cap_do': current_level,
                    'noi_dung': ' '.join(current_text).strip(),
                    'full_citation': f"Điều {current_article_id} - Luật An toàn, vệ sinh lao động số 84/2015/QH13" if current_article_id else ""
                })
                current_text = []

            current_article_id = article_match.group(1)
            current_article_title = article_match.group(2).strip()
            current_clause_id = None
            current_item_id = None
            current_level = 'dieu_intro'
            current_major_title = current_article_title
            current_text = [current_article_title]
            continue

        #Phát hiện Khoản (có nội dung trên cùng dòng)
        clause_match = patterns['clause'].match(line)
        if clause_match:
            # Lưu chunk trước đó nếu có
            if current_text:
                rows.append({
                    'chuong_id': current_chapter_id,
                    'chuong_ten': current_chapter_name,
                    'dieu_id': current_article_id,
                    'dieu_ten': current_article_title,
                    'khoan_id': current_clause_id,
                    'diem_id': current_item_id,
                    'cap_do': current_level,
                    'noi_dung': ' '.join(current_text).strip(),
                    'full_citation': f"Điều {current_article_id}, Khoản {current_clause_id} - Luật An toàn, vệ sinh lao động số 84/2015/QH13" if current_article_id and current_clause_id else ""
                })
                current_text = []

            current_clause_id = clause_match.group(1)
            current_item_id = None
            current_level = 'khoan'
            current_major_title = clause_match.group(2).strip()
            current_text = [current_major_title]
            continue

        # Phát hiện Khoản (chỉ có số, nội dung ở dòng sau)
        clause_intro_match = patterns['clause_intro'].match(line)
        if clause_intro_match:
            if current_text:
                rows.append({
                    'chuong_id': current_chapter_id,
                    'chuong_ten': current_chapter_name,
                    'dieu_id': current_article_id,
                    'dieu_ten': current_article_title,
                    'khoan_id': current_clause_id,
                    'diem_id': current_item_id,
                    'cap_do': current_level,
                    'noi_dung': ' '.join(current_text).strip(),
                    'full_citation': f"Điều {current_article_id}, Khoản {current_clause_id} - Luật An toàn, vệ sinh lao động số 84/2015/QH13" if current_article_id and current_clause_id else ""
                })
                current_text = []

            current_clause_id = clause_intro_match.group(1)
            current_item_id = None
            current_level = 'khoan'
            current_major_title = ""  # Nội dung sẽ ở dòng sau
            current_text = []
            continue

        # Phát hiện Điểm (có nội dung trên cùng dòng)
        item_match = patterns['item'].match(line)
        if item_match:
            # Lưu chunk trước đó nếu có
            if current_text:
                rows.append({
                    'chuong_id': current_chapter_id,
                    'chuong_ten': current_chapter_name,
                    'dieu_id': current_article_id,
                    'dieu_ten': current_article_title,
                    'khoan_id': current_clause_id,
                    'diem_id': current_item_id,
                    'cap_do': current_level,
                    'noi_dung': ' '.join(current_text).strip(),
                    'full_citation': f"Điều {current_article_id}, Khoản {current_clause_id}, Điểm {current_item_id} - Luật An toàn, vệ sinh lao động số 84/2015/QH13" if current_article_id and current_clause_id and current_item_id else ""
                })
                current_text = []

            current_item_id = item_match.group(1)
            current_level = 'diem'
            current_major_title = item_match.group(2).strip()
            current_text = [current_major_title]
            continue

        # Phát hiện Điểm (chỉ có ký tự, nội dung ở dòng sau)
        item_intro_match = patterns['item_intro'].match(line)
        if item_intro_match:
            if current_text:
                rows.append({
                    'chuong_id': current_chapter_id,
                    'chuong_ten': current_chapter_name,
                    'dieu_id': current_article_id,
                    'dieu_ten': current_article_title,
                    'khoan_id': current_clause_id,
                    'diem_id': current_item_id,
                    'cap_do': current_level,
                    'noi_dung': ' '.join(current_text).strip(),
                    'full_citation': f"Điều {current_article_id}, Khoản {current_clause_id}, Điểm {current_item_id} - Luật An toàn, vệ sinh lao động số 84/2015/QH13" if current_article_id and current_clause_id and current_item_id else ""
                })
                current_text = []

            current_item_id = item_intro_match.group(1)
            current_level = 'diem'
            current_major_title = ""  # Nội dung sẽ ở dòng sau
            current_text = []
            continue

        # Phát hiện câu bổ sung (khoan_bo_sung) - đơn giản hóa: nếu có cụm từ đặc trưng
        # Nếu đang ở trong khoản và gặp dòng bắt đầu bằng "Trường hợp", "Căn cứ", ... thì coi là khoan_bo_sung
        if current_level in ['khoan', 'diem'] and line:
            # Kiểm tra nếu dòng bắt đầu bằng các từ khóa đặc trưng của khoản bổ sung
            bo_sung_keywords = ["Trường hợp", "Căn cứ", "Chính phủ", "Nhà nước", "Đối với", "Sau khi", "Việc", "Mục", "Bộ trưởng"]
            if any(line.startswith(keyword) for keyword in bo_sung_keywords):
                # Lưu chunk hiện tại trước khi chuyển sang khoan_bo_sung
                if current_text:
                    rows.append({
                        'chuong_id': current_chapter_id,
                        'chuong_ten': current_chapter_name,
                        'dieu_id': current_article_id,
                        'dieu_ten': current_article_title,
                        'khoan_id': current_clause_id,
                        'diem_id': current_item_id,
                        'cap_do': current_level,
                        'noi_dung': ' '.join(current_text).strip(),
                        'full_citation': f"Điều {current_article_id}, Khoản {current_clause_id} - Luật An toàn, vệ sinh lao động số 84/2015/QH13" if current_article_id and current_clause_id else ""
                    })
                    current_text = []
                current_level = 'khoan_bo_sung'
                current_major_title = line
                current_text = [line]
                continue

        # Nếu không khớp với bất kỳ pattern nào, thêm dòng vào nội dung hiện tại
        if current_text is not None:
            # Nếu là dòng đầu tiên và chưa có nội dung chính (trường hợp clause_intro hoặc item_intro), gán làm nội dung chính
            if not current_text and current_major_title:
                current_text = [current_major_title + " " + line]
                current_major_title = ""  # Reset sau khi đã dùng
            else:
                current_text.append(line)

    # Lưu chunk cuối cùng
    if current_text:
        rows.append({
            'chuong_id': current_chapter_id,
            'chuong_ten': current_chapter_name,
            'dieu_id': current_article_id,
            'dieu_ten': current_article_title,
            'khoan_id': current_clause_id,
            'diem_id': current_item_id,
            'cap_do': current_level,
            'noi_dung': ' '.join(current_text).strip(),
            'full_citation': f"Điều {current_article_id}, Khoản {current_clause_id} - Luật An toàn, vệ sinh lao động số 84/2015/QH13" if current_article_id and current_clause_id else ""
        })


    # Tạo DataFrame
    df = pd.DataFrame(rows)

    # Thêm cột ten_van_ban, ngay_ban_hanh, co_quan_ban_hanh, so_hieu
    df['ten_van_ban'] = 'Luật An toàn, vệ sinh lao động 2015'
    df['ngay_ban_hanh'] = 'ngày 25 tháng 06 năm 2015'
    df['co_quan_ban_hanh'] = 'Quốc Hội'
    df['so_hieu'] = 'Luật số: 84/2015/QH13'

    # Chuẩn hóa lại full_citation cho các dòng
    for idx, row in df.iterrows():
        if row['cap_do'] == 'dieu_intro':
            df.at[idx, 'full_citation'] = f"Điều {row['dieu_id']} - Luật An toàn, vệ sinh lao động số 84/2015/QH13"
        elif row['cap_do'] == 'khoan':
            df.at[idx, 'full_citation'] = f"Điều {row['dieu_id']}, Khoản {row['khoan_id']} - Luật An toàn, vệ sinh lao động số 84/2015/QH13"
        elif row['cap_do'] == 'diem':
            df.at[idx, 'full_citation'] = f"Điều {row['dieu_id']}, Khoản {row['khoan_id']}, Điểm {row['diem_id']} - Luật An toàn, vệ sinh lao động số 84/2015/QH13"
        elif row['cap_do'] == 'khoan_bo_sung':
            df.at[idx, 'full_citation'] = f"Điều {row['dieu_id']} - Luật An toàn, vệ sinh lao động số 84/2015/QH13"

    return df

# Parse văn bản
if full_text_standardized:
    df_chunks = parse_law_text_to_chunks(full_text_standardized)
    print(f"Đã tạo {len(df_chunks)} chunks.")
    print("\nMẫu dữ liệu:")
    print(df_chunks[['chuong_id', 'dieu_id', 'khoan_id', 'diem_id', 'cap_do', 'noi_dung']].head(10))

    # Lưu file CSV
    #df_chunks.to_csv('law_dataset_chunks.csv', index=False, encoding='utf-8-sig')
    # Thay vì lưu tạm thời:
# df_chunks.to_csv('law_dataset_chunks.csv', index=False, encoding='utf-8-sig')

# Hãy lưu thẳng vào thư mục data trên Drive của bạn:
    file_path_csv = '/content/drive/MyDrive/Colab Notebooks/NLP/law_retrieval/data/law_dataset_chunks.csv'
    df_chunks.to_csv(file_path_csv, index=False, encoding='utf-8-sig', sep='|')
    print(f"Đã lưu file thẳng vào Drive: {file_path_csv}")

    print("\nĐã lưu file law_dataset_chunks.csv")
else:
    print("Không thể parse văn bản do lỗi đọc file.")





Đã tạo 478 chunks.

Mẫu dữ liệu:
  chuong_id dieu_id khoan_id diem_id      cap_do  \
0      None    None     None    None        None   
1      None       1     None    None  dieu_intro   
2      None       2     None    None  dieu_intro   
3      None       2        1    None       khoan   
4      None       2        2    None       khoan   
5      None       2        3    None       khoan   
6      None       2        4    None       khoan   
7      None       2        5    None       khoan   
8      None       2        6    None       khoan   
9      None       3     None    None  dieu_intro   

                                            noi_dung  
0  QUỐC HỘI CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độ...  
1  Phạm vi điều chỉnh Luật này quy định việc bảo ...  
2                                  Đối tượng áp dụng  
3  Người lao động làm việc theo hợp đồng lao động...  
4  Cán bộ, công chức, viên chức, người thuộc lực ...  
5  Người lao động làm việc không theo hợp đồng la...  
6  Người 